# M2C Lab — Semantic Chunking + RBAC RAG in SageMaker

**Track:** Applied Agent Engineering Foundations  
**Module fit:** M2 — RAG Fundamentals & Retrieval Patterns  
**Derived from your upload:** semantic chunking + salary RBAC example

## What learners will understand
1. Why semantic chunking differs from fixed chunking
2. How retrieval can be filtered by business rules
3. Why access control belongs inside enterprise RAG design
4. How to capture audit information in notebook-native tables


## Classroom framing

A lot of RAG demos retrieve *relevant* context.  
Enterprise RAG must retrieve context that is both:
- relevant
- authorized

That is the point of this lab.


In [ ]:

# Uncomment only if your image is missing dependencies.
# %pip install -q boto3 pandas sentence-transformers faiss-cpu langchain-experimental langchain-community python-docx


## Step 1 — Configuration


In [ ]:

from __future__ import annotations

import io
import os
import re
from pathlib import Path

import boto3
import faiss
import numpy as np
import pandas as pd

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
BEDROCK_MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"

USE_S3 = False
S3_BUCKET = "replace-me"
POLICY_PREFIX = "replace-me/policies/"
SALARY_KEY = "replace-me/salary.csv"

TOP_K = 4
LOCAL_WORK_DIR = Path("./m2_rbac_rag_artifacts")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)


## Step 2 — Load policy text and salary roster

For class stability, the notebook can run with the built-in toy data.
Replace with S3 paths when your real lab data is ready.


In [ ]:

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

def read_csv_from_s3(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_csv(io.BytesIO(body))

def list_text_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".txt"):
                keys.append(key)
    return keys

def read_s3_text(bucket: str, key: str) -> str:
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8", errors="ignore")

if USE_S3:
    policy_docs = [{"source": key, "text": read_s3_text(S3_BUCKET, key)} for key in list_text_keys(S3_BUCKET, POLICY_PREFIX)]
    salary_df = read_csv_from_s3(S3_BUCKET, SALARY_KEY)
else:
    policy_docs = [
        {"source": "leave_policy.txt", "text": "Employees should apply planned leave in advance. Leave may be declined if there is a critical business dependency."},
        {"source": "travel_policy.txt", "text": "Travel reimbursements require manager approval and proper documentation."},
        {"source": "security_policy.txt", "text": "Access should follow least privilege and need-to-know principles."},
    ]
    salary_df = pd.DataFrame([
        {"employee_id": "EMP-0001", "employee_name": "Asha Rao", "level": "L3", "role": "Engineer", "annual_salary_inr": 1200000},
        {"employee_id": "EMP-0002", "employee_name": "Ravi Menon", "level": "L5", "role": "Manager", "annual_salary_inr": 2400000},
        {"employee_id": "EMP-0003", "employee_name": "Neha Singh", "level": "L7", "role": "CEO", "annual_salary_inr": 5000000},
    ])

display(pd.DataFrame(policy_docs))
display(salary_df)


## Step 3 — Build policy chunks and salary documents


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

try:
    from langchain_experimental.text_splitter import SemanticChunker
    from langchain_core.documents import Document as LCDocument

    semantic_chunker = SemanticChunker(embedding_model)

    policy_chunk_rows = []
    for doc in policy_docs:
        lc_docs = [LCDocument(page_content=doc["text"], metadata={"source": doc["source"]})]
        semantic_chunks = semantic_chunker.split_documents(lc_docs)

        for i, ch in enumerate(semantic_chunks, start=1):
            policy_chunk_rows.append({
                "doc_type": "policy",
                "source": doc["source"],
                "doc_id": doc["source"],
                "chunk_id": f'{doc["source"]}::chunk_{i:03d}',
                "text": ch.page_content,
                "employee_id": None,
                "level_num": None
            })
    chunking_mode = "semantic"
except Exception:
    def fixed_chunks(text: str, size: int = 300) -> list[str]:
        return [text[i:i+size] for i in range(0, len(text), size)]

    policy_chunk_rows = []
    for doc in policy_docs:
        for i, ch in enumerate(fixed_chunks(doc["text"]), start=1):
            policy_chunk_rows.append({
                "doc_type": "policy",
                "source": doc["source"],
                "doc_id": doc["source"],
                "chunk_id": f'{doc["source"]}::chunk_{i:03d}',
                "text": ch,
                "employee_id": None,
                "level_num": None
            })
    chunking_mode = "fixed_fallback"

salary_docs = []
for r in salary_df.itertuples():
    level_num = int(re.findall(r"\d+", r.level)[0])
    salary_docs.append({
        "doc_type": "salary",
        "source": "salary_roster.csv",
        "doc_id": r.employee_id,
        "chunk_id": f"{r.employee_id}::salary",
        "text": f"Employee ID: {r.employee_id}. Employee Name: {r.employee_name}. Level: {r.level}. Role: {r.role}. Annual Salary INR: {int(r.annual_salary_inr)}.",
        "employee_id": r.employee_id,
        "level_num": level_num
    })

corpus_df = pd.DataFrame(policy_chunk_rows + salary_docs)
print("Chunking mode:", chunking_mode)
display(corpus_df)


## Step 4 — Create the retrieval index


In [ ]:

embeddings = embedding_model.encode(
    corpus_df["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False
)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings, dtype="float32"))

print("Indexed rows:", index.ntotal)


## Step 5 — RBAC rules

Salary access rule used in this teaching notebook:
- L1–L4: can see only own salary
- L5–L6: can see own salary and levels below them
- L7: can see all salaries
- non-salary questions can use policy docs only


In [ ]:

def parse_level(level: str) -> int:
    return int(re.findall(r"\d+", level)[0])

def allowed_salary_ids(requester_level: int, requester_employee_id: str | None) -> set[str]:
    if requester_level >= 7:
        return set(salary_df["employee_id"].tolist())

    if requester_level <= 4:
        return {requester_employee_id} if requester_employee_id else set()

    requester_row = salary_df.loc[salary_df["employee_id"] == requester_employee_id]
    if len(requester_row) == 0:
        return set()
    requester_level_num = parse_level(requester_row.iloc[0]["level"])

    allowed = set(
        salary_df.loc[
            salary_df["level"].apply(parse_level) < requester_level_num,
            "employee_id"
        ].tolist()
    )
    allowed.add(requester_employee_id)
    return allowed

def is_salary_question(q: str) -> bool:
    return bool(re.search(r"\b(salary|pay|compensation|ctc)\b", q.lower()))

def retrieve_with_rbac(query: str, requester_level: int, requester_employee_id: str | None, top_k: int = TOP_K) -> pd.DataFrame:
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), min(12, len(corpus_df)))

    allowed_ids = allowed_salary_ids(requester_level, requester_employee_id)
    rows = []

    for score, idx_ in zip(scores[0], idxs[0]):
        row = corpus_df.iloc[int(idx_)].to_dict()

        if row["doc_type"] == "salary":
            if row["employee_id"] not in allowed_ids:
                continue

        if row["doc_type"] == "policy" and is_salary_question(query):
            # keep policy snippets too if they are relevant to the salary query context
            pass

        row["score"] = float(score)
        rows.append(row)

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

audit_test = retrieve_with_rbac(
    query="What is EMP-0001 salary?",
    requester_level=5,
    requester_employee_id="EMP-0002",
    top_k=TOP_K
)
display(audit_test)


## Step 6 — Bedrock answer generation over authorized context only


In [ ]:

def call_bedrock(prompt: str, model_id: str = BEDROCK_MODEL_ID) -> str:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 250, "temperature": 0.1}
    )
    return response["output"]["message"]["content"][0]["text"]

def answer_secure_query(query: str, requester_level: int, requester_employee_id: str | None) -> tuple[pd.DataFrame, str]:
    authorized_df = retrieve_with_rbac(query, requester_level, requester_employee_id, top_k=TOP_K)

    if len(authorized_df) == 0:
        return authorized_df, "No authorized evidence found for this request."

    context = "\n\n".join([f"[{i+1}] {t}" for i, t in enumerate(authorized_df["text"].tolist())])

    prompt = f'''
You are an enterprise HR assistant.
Answer only from the authorized context below.
If the answer is not present, say that you do not know from the authorized context.

Authorized context:
{context}

Question:
{query}

Answer:
'''.strip()

    answer = call_bedrock(prompt)
    return authorized_df, answer

authorized_df, secure_answer = answer_secure_query(
    query="What is EMP-0001 salary?",
    requester_level=5,
    requester_employee_id="EMP-0002"
)

display(authorized_df)
print(secure_answer)


## Step 7 — Audit table

This is the notebook-native governance pattern for class:
- requester identity
- level
- question
- authorized evidence
- final answer


In [ ]:

audit_log_df = pd.DataFrame([{
    "requester_level": 5,
    "requester_employee_id": "EMP-0002",
    "question": "What is EMP-0001 salary?",
    "authorized_rows": len(authorized_df),
    "authorized_sources": " | ".join(authorized_df["chunk_id"].tolist()) if len(authorized_df) else None,
    "timestamp": pd.Timestamp.utcnow()
}])

display(audit_log_df)


## Mini-hack — M2 secure retrieval challenge

Learners can do one of these:
1. Add a new rule for department-based filtering.
2. Block bulk salary questions like “show me everyone’s salaries” for L5.
3. Add a column `decision` with `allowed` or `denied`.
4. Compare fixed chunking vs semantic chunking for policy docs.

## Reflection questions
- Why is relevance alone not enough in enterprise RAG?
- At what layer should RBAC be enforced?
- What evidence would an auditor ask for after a sensitive answer?
